In [8]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
import pickle
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import re
import json

load_dotenv()
#api_key = os.getenv("OPENAI_API_KEY")
#client = OpenAI(api_key=api_key)

True

# Summarization with GPT5Nano

In [ ]:
def extract_text_from_response(resp):
    texts = []

    for item in resp.output:
        # We only care about assistant messages
        if hasattr(item, "type") and item.type == "message":
            for content in item.content:
                if hasattr(content, "type") and content.type == "output_text":
                    texts.append(content.text)

    return "\n".join(texts).strip()

In [ ]:
def parse_sentiment_output(text):
    patterns = {
        "Impact": r"Impact:\s*(Bullish|Bearish|Neutral)",
        "ExpectedPctMove": r"Expected % Move:\s*([+-]?\d+)",
        "TimeHorizon": r"Time Horizon:\s*([^\n]+)",
        "Confidence": r"Confidence:\s*(\d+)",
        "Reason": r"Reason:\s*(.+)",
    }

    result = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, text)
        result[key] = match.group(1).strip() if match else None

    # Type coercion
    if result["ExpectedPctMove"] is not None:
        result["ExpectedPctMove"] = int(result["ExpectedPctMove"])

    if result["Confidence"] is not None:
        result["Confidence"] = int(result["Confidence"])

    return result

In [ ]:
def summarize_filing(text):
    prompt = f"""
Summarize the following NSE corporate filing in 3–4 bullet points.

Focus on:
- What happened
- Quantitative details (amounts, capacity, dates)
- Impact on business or revenue

Ignore legal boilerplate, greetings, addresses.

FILING:
{text}
"""

    resp = client.responses.create(
        model="gpt-5-nano",
        input=prompt,
        max_output_tokens=300,
        reasoning={"effort": "minimal"},
    )

    return resp

In [ ]:
def summarize_pulse_article(text):
    prompt = f"""
Summarize the following market news in 2–3 bullet points.

Focus on:
- The concrete event or trigger (if any)
- Which companies or sector are directly affected
- Ignore opinions, price targets, and generic market commentary

If there is no actionable event, say:
"Non-actionable market commentary."

NEWS:
{text}
"""

    resp = client.responses.create(
        model="gpt-5-nano",
        input=prompt,
        max_output_tokens=300,
        reasoning={"effort": "minimal"},
    )

    return resp

In [ ]:
def classify_sentiment(summary):
    prompt = f"""
You are a market analyst evaluating the stock impact of a corporate event.

Based ONLY on the event summary below, provide a structured assessment.

Return the answer in the following format:

Impact: Bullish | Bearish | Neutral
Expected % Move: <single number, positive or negative, e.g. +4 or -3>
Time Horizon: <1–2 days | 3–5 days | 1–2 weeks>
Confidence: <integer from 0 to 100>
Reason: <one concise sentence explaining the impact>

Guidelines:
- Expected % Move should reflect a realistic short-term move for a liquid Indian F&O stock.
- Time Horizon refers to when most of the move is likely to materialize.
- Confidence reflects how certain the impact is, given the information quality and clarity.
- Do NOT invent facts or numbers not implied by the event.

EVENT SUMMARY:
{summary}
"""

    resp = client.responses.create(
        model="gpt-5-nano",
        input=prompt,
        max_output_tokens=400,
        reasoning={"effort": "low"},
    )

    return resp


In [6]:
with open("./Data/NSE/nse_news.pkl", "rb") as f:
    nse_news = pickle.load(f)
with open("./Data/Pulse/pulse_news.pkl", "rb") as f:
    pulse_news = pickle.load(f)

In [7]:
#Drop duplicates
pulse_news = pulse_news.drop_duplicates(subset=["Complete_Article"])
nse_news = nse_news.drop_duplicates(subset=["ATTACHMENT"])

In [ ]:
batch_file = Path("pulse_batch.jsonl")

with batch_file.open("w") as f:
    for i, row in pulse_news.iterrows():
        prompt = f"""
        You are a market analyst evaluating the stock impact of a corporate event.

        Based ONLY on the event summary below, provide a structured assessment.

        Return the answer in the following format:

        Impact: Bullish | Bearish | Neutral
        Expected % Move: <single number, positive or negative, e.g. +4 or -3>
        Time Horizon: <1–2 days | 3–5 days | 1–2 weeks>
        Confidence: <integer from 0 to 100>
        Reason: <one concise sentence explaining the impact>

        Guidelines:
        - Expected % Move should reflect a realistic short-term move for a liquid Indian F&O stock.
        - Time Horizon refers to when most of the move is likely to materialize.
        - Confidence reflects how certain the impact is, given the information quality and clarity.
        - Do NOT invent facts or numbers not implied by the event.

        EVENT SUMMARY:
        {row['Summary']}
        """

        request = {
            "custom_id": f"pulse_{i}",
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": "gpt-5-nano",
                "input": prompt,
                "max_output_tokens": 200,
                "reasoning": {"effort": "none"}
            }
        }

        f.write(json.dumps(request) + "\n")


In [ ]:
#Update completion window
batch = client.batches.create(
    input_file=batch_file,
    endpoint="/v1/responses",
    completion_window="24h"
)

print("Batch ID:", batch.id)

# Use Batch API

In [ ]:
batch_id = batch.id

while True:
    batch = client.batches.retrieve(batch_id)
    print("Status:", batch.status)

    if batch.status in ["completed", "failed"]:
        break

    time.sleep(30)


In [ ]:
result_file_id = batch.output_file_id
content = client.files.content(result_file_id).read()

lines = content.decode("utf-8").splitlines()
results = [json.loads(line) for line in lines]


In [ ]:
result_map = {}

for r in results:
    custom_id = r["custom_id"]
    text = extract_text_from_response(r["response"])
    result_map[custom_id] = text

for i in pulse_news.index:
    key = f"pulse_{i}"
    pulse_news.loc[i, "SentimentRaw"] = result_map.get(key)
